# 2. RQ1: Adoção e autoria por agentes

**Pergunta:** o formato está se espalhando? E quem escreve as skills: pessoas ou os próprios agentes de IA?

**Por que importa:** se boa parte das instruções que os agentes seguem foi escrita por agentes, o ecossistema se retroalimenta, e ninguém revisa um texto que nenhuma pessoa escreveu inteiramente.

**Como medimos:** (1) contamos novas skills por mês desde o lançamento (out/2025); (2) olhamos o primeiro commit de cada skill: se ele tem um trailer `Co-authored-by: Claude` (ou Cursor, Copilot…) ou foi feito por uma conta bot, contamos como escrita por assistente. A margem de erro é o intervalo de confiança de Wilson a 95%.

---

### English summary

**Methodology step 3: quantitative descriptive analysis.**

* RQ1.1 How fast did the format spread after October 2025?
* RQ1.2 What share of skills were first committed with an AI assistant credited as co-author (a `Co-authored-by` trailer) or by a bot account?

Proportions are reported with 95% Wilson confidence intervals.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])
from statsmodels.stats.proportion import proportion_confint

In [ ]:
skills = clean.strict(clean.load_skills())
cov = adoption.history_coverage(skills)
complete = cov.loc[cov["share"] >= 0.9, "location_class"].tolist()
print("location classes with complete history:", complete)

## 2.1 Novas skills por mês
**Objetivo:** medir o ritmo de adoção. Usamos só `canonical`, o único local com histórico completo (notebook 01, §1.5); misturar locais amostrados distorceria a curva.

In [ ]:
std = skills[skills["location_class"].isin(complete)]
cohorts = adoption.monthly_cohorts(std)
ax = cohorts.set_index("first_month")["new_skills"].plot(marker=".", figsize=(6, 2.6))
ax.set_ylabel("new distinct skills"); plt.show()
cohorts

**O que os dados mostram:** de **953** novas skills em out/2025 para **45.672** em jun/2026, cerca de **48 vezes** em oito meses. O crescimento acelera até mar/2026 e depois se estabiliza em ~40–45 mil por mês. Julho cai porque o snapshot foi coletado no meio do mês, não por queda de adoção.

## 2.2 Autoria por assistente
**Objetivo:** estimar que fração das skills foi criada com um agente de IA registrado como coautor.

In [ ]:
h = skills[skills["has_history"]].copy()
h["ai_first"] = h["first_commit_ai"].notna() | h["first_commit_bot"]
k, n = int(h["ai_first"].sum()), len(h)
lo, hi = proportion_confint(k, n, method="wilson")
print(f"first commit by an assistant or bot: {k:,}/{n:,} = {k/n:.1%} (95% CI {lo:.1%}-{hi:.1%})")

**O que os dados mostram:** **31,9%** (IC 95%: 31,8–32,1%) das skills com histórico nasceram em um commit assinado por um assistente ou bot. **Cerca de uma em cada três skills foi escrita com um agente.**

In [ ]:
h["first_commit_ai"].value_counts().rename("skills").to_frame()

**O que os dados mostram:** o Claude responde por **95%** desses casos (116.609), seguido de longe por Cursor (3.311) e Copilot (2.435). Isso reflete tanto a origem do formato (Claude Code) quanto o fato de o Claude adicionar o trailer por padrão; outras ferramentas podem estar sub-representadas.

In [ ]:
by_month = h.groupby("first_month")["ai_first"].agg(skills="size", share="mean")
ax = by_month["share"].plot(marker=".", figsize=(6, 2.6)); ax.set_ylabel("share assistant-authored"); plt.show()
by_month

**O que os dados mostram:** a fração fica entre **29% e 40%** em todos os meses, sem tendência clara de queda ou alta. A autoria por agentes não é uma fase inicial: acompanha o crescimento do ecossistema.

## Conclusão da RQ1
* O ecossistema cresceu cerca de **48×** em oito meses.
* **Cerca de 1 em cada 3** skills foi criada com um agente como coautor, quase sempre o Claude, e essa fração é estável no tempo.

**Limitações:** é um **limite inferior** (só conta quem manteve o trailer `Co-authored-by`); vale para as 396.247 skills com histórico, concentradas em `.claude/skills/`.

**Próximo passo:** se as skills são criadas tão rápido, de onde vêm? O notebook 03 mostra que boa parte vem de cópia.